# Logistic Regression Pipeline

One of two independent training approaches for predicting progression to Alzheimer's disease from Mild Cognitive Impairment. This notebook works from a 2-feature subset (chosen via exhaustive search) and Logistic Regression; see `svm_pipeline.ipynb` for the other approach (all 7 features, SVM). `main.ipynb` loads the models both notebooks save here and averages their predictions as a final ensemble.

# Determine the best parameters
### Load the data, taking care of Nan values
### 

We have a problem, some of the Mild Cognitive Impairments have Nan values. So we are going to replace the Nan values with the most common value or mode which is 'No'

# The best parameters


In [1]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score

In [2]:
import pandas as pd

# 1. Load data, keep only patients with a known outcome (MCI patients)
df = pd.read_csv('data/plasma_lipidomics.csv')
mci = df[df["Progression to Alzheimer's Disease"].notna()].copy()

# 2. Fill missing numeric values with the column median
numeric_cols = ['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)',
                 'CSF Phosphorylated tau (pg/mL)']
for col in numeric_cols:
    median_val = mci[col].median()
    mci[col] = mci[col].fillna(median_val)

# 3. Fill missing categorical values with the most common value
mode_val = mci['APOE4'].mode()[0]
mci['APOE4'] = mci['APOE4'].fillna(mode_val)

# 4. Convert categorical text columns to numeric (0/1)
mci['Sex'] = (mci['Sex'] == 'Male').astype(int)          # Male=1, Female=0
mci['APOE4'] = (mci['APOE4'] == 'Yes').astype(int)        # carries APOE4 allele=1, no=0
mci['Target'] = (mci["Progression to Alzheimer's Disease"] == 'Yes').astype(int)

# 5. Final feature set + target
feature_cols = numeric_cols + ['Sex', 'APOE4']
X = mci[feature_cols]
y = mci['Target']

In [3]:
from sklearn.metrics import precision_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

def evaluate_performance(terms, X, y, verbose=False):
    # ---- 5-fold cross-validation, honest out-of-fold predictions ----
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])

    y_pred = cross_val_predict(model, X, y, cv=cv, method='predict')
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]

    # ---- Confusion matrix ----
    cm = confusion_matrix(y, y_pred, labels=[0, 1])

    if verbose:
        display_terms = ' '.join(terms)
        print(f"Predicting 'Progression to Alzheimer's Disease' from {display_terms}\n")
        print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
        print(classification_report(y, y_pred, target_names=['No progression', 'Progressed'], digits=3))
        print('Accuracy:', accuracy_score(y, y_pred))
        print('AUC:', roc_auc_score(y, y_proba))

    # ---- Extract metrics ----
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y, y_pred)
    accuracy = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'features': terms,
        'n_features': len(terms),
        'accuracy': accuracy,
        'precision': precision,
        'auc': auc,
        'false_negatives': fn,
        'false_negative_rate': fnr,
    }

In [4]:
import itertools

feature_list = list(X.columns)
all_combos = []
for r in range(1, len(feature_list) + 1):
    all_combos.extend(itertools.combinations(feature_list, r))

results = []
for combo in all_combos:
    cols = list(combo)
    print(cols)
    result = evaluate_performance(cols, X[cols], y)
    results.append(result)

results_df = pd.DataFrame(results)

# ---- Filter: FNR <= 0.20, sort by accuracy desc, AUC as tiebreaker ----
best_df = (
    results_df[results_df['false_negative_rate'] <= 0.20]
    .sort_values(by=['accuracy', 'auc'], ascending=[False, False])
    .reset_index(drop=True)
)

print(f"{len(best_df)} of {len(results_df)} combinations meet the FNR <= 0.20 threshold\n")
print(best_df.head(10))

['Age']
['MMSE']
['CSF Amyloid (pg/mL)']


['CSF Total tau (pg/mL)']
['CSF Phosphorylated tau (pg/mL)']
['Sex']


['APOE4']
['Age', 'MMSE']
['Age', 'CSF Amyloid (pg/mL)']


['Age', 'CSF Total tau (pg/mL)']
['Age', 'CSF Phosphorylated tau (pg/mL)']
['Age', 'Sex']


['Age', 'APOE4']
['MMSE', 'CSF Amyloid (pg/mL)']
['MMSE', 'CSF Total tau (pg/mL)']


...

['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'APOE4']
['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'Sex', 'APOE4']


['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'Sex', 'APOE4']
['Age', 'MMSE', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'Sex', 'APOE4']


['Age', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'Sex', 'APOE4']


['MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'Sex', 'APOE4']


['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'Sex', 'APOE4']
8 of 127 combinations meet the FNR <= 0.20 threshold

                                            features  n_features  accuracy  \
0  [CSF Amyloid (pg/mL), CSF Phosphorylated tau (...           2  0.730337   
1  [CSF Amyloid (pg/mL), CSF Phosphorylated tau (...           3  0.719101   
2                              [CSF Amyloid (pg/mL)]           1  0.719101   
3                         [Age, CSF Amyloid (pg/mL)]           2  0.719101   
4                         [CSF Amyloid (pg/mL), Sex]           2  0.719101   
5                    [Age, CSF Amyloid (pg/mL), Sex]           3  0.719101   
6  [Age, CSF Amyloid (pg/mL), CSF Phosphorylated ...           3  0.707865   
7  [Age, CSF Amyloid (pg/mL), CSF Phosphorylated ...           4  0.707865   

   precision       auc  false_negatives  false_negative_rate  
0   0.709091  0.779382                8             0.170213  
1  

In [5]:
best_df.columns

Index(['features', 'n_features', 'accuracy', 'precision', 'auc',
       'false_negatives', 'false_negative_rate'],
      dtype='str')

In [6]:
best = best_df.iloc[0]

In [7]:
best.features

['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']

In [8]:
best.T

features               [CSF Amyloid (pg/mL), CSF Phosphorylated tau (...
n_features                                                             2
accuracy                                                        0.730337
precision                                                       0.709091
auc                                                             0.779382
false_negatives                                                        8
false_negative_rate                                             0.170213
Name: 0, dtype: object

# best classification algorithm

In [9]:
from sklearn.metrics import precision_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import pandas as pd

def evaluate_pipeline(terms, X, y, model, verbose=False):
    # ---- 5-fold cross-validation, honest out-of-fold predictions ----
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    y_pred = cross_val_predict(model, X, y, cv=cv, method='predict')
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]

    # ---- Confusion matrix ----
    cm = confusion_matrix(y, y_pred, labels=[0, 1])

    if verbose:
        display_terms = ' '.join(terms)
        print(f"Predicting 'Progression to Alzheimer's Disease' from {display_terms}\n")
        print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
        print(classification_report(y, y_pred, target_names=['No progression', 'Progressed'], digits=3))
        print('Accuracy:', accuracy_score(y, y_pred))
        print('AUC:', roc_auc_score(y, y_proba))

    # ---- Extract metrics ----
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y, y_pred)
    accuracy = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'features': terms,
        'n_features': len(terms),
        'accuracy': accuracy,
        'precision': precision,
        'auc': auc,
        'false_negatives': fn,
        'false_negative_rate': fnr,
    }

In [10]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest':        RandomForestClassifier(n_estimators=300, random_state=42),
    'SVM (RBF)':            Pipeline([('sc', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]),
    'Gradient Boosting':    GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors':  Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier())]),
    'LDA':                  Pipeline([('sc', StandardScaler()), ('clf', LinearDiscriminantAnalysis())]),
    'Naive Bayes':          Pipeline([('sc', StandardScaler()), ('clf', GaussianNB())]),
}

best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
X_best = X[best_features]

model_results = []
for name, model in models.items():
    result = evaluate_pipeline(best_features, X_best, y, model)
    result['model'] = name
    model_results.append(result)

model_results_df = pd.DataFrame(model_results)

# ---- Filter: FNR <= 0.20, sort by accuracy desc, AUC as tiebreaker ----
best_models_df = (
    model_results_df[model_results_df['false_negative_rate'] <= 0.20]
    .sort_values(by=['accuracy', 'auc'], ascending=[False, False])
    .reset_index(drop=True)
)

print(f"{len(best_models_df)} of {len(model_results_df)} models meet the FNR <= 0.20 threshold\n")
print(best_models_df[['model', 'accuracy', 'precision', 'auc', 'false_negatives', 'false_negative_rate']])

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 a

3 of 7 models meet the FNR <= 0.20 threshold

                 model  accuracy  precision       auc  false_negatives  \
0  Logistic Regression  0.730337   0.709091  0.779382                8   
1                  LDA  0.707865   0.684211  0.779382                8   
2          Naive Bayes  0.696629   0.672414  0.790780                8   

   false_negative_rate  
0             0.170213  
1             0.170213  
2             0.170213  


In [11]:
## logistic regression

In [12]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import make_scorer, fbeta_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
X_best = X[best_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])

# Modest grid given the small sample size (89 patients)
param_grid = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l2'],
    'clf__class_weight': [None, 'balanced'],
}

f2_scorer = make_scorer(fbeta_score, beta=2)

scoring_options = {
    'recall': 'recall',
    'f2': f2_scorer,
}

def run_grid_search(scoring_name, scoring):
    grid = GridSearchCV(pipe, param_grid, scoring=scoring, cv=cv, n_jobs=1)
    grid.fit(X_best, y)

    best_model = grid.best_estimator_

    # Evaluate the winning hyperparameters with honest out-of-fold predictions
    y_pred = cross_val_predict(best_model, X_best, y, cv=cv, method='predict')
    y_proba = cross_val_predict(best_model, X_best, y, cv=cv, method='predict_proba')[:, 1]

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

    return {
        'scoring': scoring_name,
        'best_params': grid.best_params_,
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred),
        'auc': roc_auc_score(y, y_proba),
        'false_negatives': int(fn),
        'false_negative_rate': fnr,
    }

results = [run_grid_search(name, scoring) for name, scoring in scoring_options.items()]
comparison_df = pd.DataFrame(results)
print(comparison_df)

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

  scoring                                        best_params  accuracy  \
0  recall  {'clf__C': 0.01, 'clf__class_weight': None, 'c...  0.595506   
1      f2  {'clf__C': 10, 'clf__class_weight': None, 'clf...  0.741573   

   precision       auc  false_negatives  false_negative_rate  
0   0.579710  0.777356                7             0.148936  
1   0.722222  0.776849                8             0.170213  


/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/fazuskazoo/.local/lib/python3.14/s

F2 wins 
'clf', LogisticRegression(C=10, class_weight=None, penalty='l2', max_iter=1000

## Train the model

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, roc_auc_score
)
import pandas as pd

best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
X_best = X[best_features]

# ---- Stratified train/test split (80/20), preserves class balance ----
X_train, X_test, y_train, y_test = train_test_split(
    X_best, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train size: {len(X_train)}  ({y_train.sum()} Yes / {len(y_train) - y_train.sum()} No)")
print(f"Test size:  {len(X_test)}  ({y_test.sum()} Yes / {len(y_test) - y_test.sum()} No)")

# ---- Build and fit the final model ----
final_model = Pipeline([
    ('sc', StandardScaler()),
    ('clf', LogisticRegression(C=10, class_weight=None, penalty='l2', max_iter=1000))
])

final_model.fit(X_train, y_train)

# ---- Evaluate on the held-out test set ----
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
print(classification_report(y_test, y_pred, target_names=['No progression', 'Progressed'], digits=3))

tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp) if (fn + tp) > 0 else float('nan')

test_result = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'auc': roc_auc_score(y_test, y_proba),
    'false_negatives': int(fn),
    'false_negative_rate': fnr,
}
print(test_result)

Train size: 71  (37 Yes / 34 No)
Test size:  18  (10 Yes / 8 No)
             Pred: No  Pred: Yes
Actual: No          6          2
Actual: Yes         2          8
                precision    recall  f1-score   support

No progression      0.750     0.750     0.750         8
    Progressed      0.800     0.800     0.800        10

      accuracy                          0.778        18
     macro avg      0.775     0.775     0.775        18
  weighted avg      0.778     0.778     0.778        18

{'accuracy': 0.7777777777777778, 'precision': 0.8, 'auc': 0.85, 'false_negatives': 2, 'false_negative_rate': np.float64(0.2)}


/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## The model

In [14]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

best_features = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
X_best = X[best_features]

# ---- Build and fit on the FULL dataset (all 89 patients) ----
deployable_model = Pipeline([
    ('sc', StandardScaler()),
    ('clf', LogisticRegression(C=10, class_weight=None, penalty='l2', max_iter=1000))
])

deployable_model.fit(X_best, y)

# ---- Save to disk ----
joblib.dump(deployable_model, 'alzheimers_progression_model.joblib')
print("Model saved to alzheimers_progression_model.joblib")

# ---- Also save the feature list, since the model needs columns in this exact order ----
joblib.dump(best_features, 'alzheimers_progression_model_features.joblib')

Model saved to alzheimers_progression_model.joblib


/home/fazuskazoo/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


['alzheimers_progression_model_features.joblib']

# Load the Model

In [15]:
import joblib
import pandas as pd

model = joblib.load('alzheimers_progression_model.joblib')
features = joblib.load('alzheimers_progression_model_features.joblib')

# new_patients must be a DataFrame with the same columns, in the same order
def predict_progression(new_patients_df):
    X_new = new_patients_df[features]
    predictions = model.predict(X_new)
    probabilities = model.predict_proba(X_new)[:, 1]
    return pd.DataFrame({
        'predicted_progression': predictions,
        'progression_probability': probabilities
    }, index=new_patients_df.index)

# example:
# results = predict_progression(new_patients_df)

In [16]:
import pandas as pd

feature_cols = ['CSF Amyloid (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']

# Raw array: [amyloid, p-tau] per patient
test_array = [
    [368.2, 119.2],   # High-risk profile
    [604.0, 63.5],    # Median profile
    [1062.0, 37.1],   # Low-risk profile
    [257.0, 512.0],   # Extreme AD-like (observed min amyloid + max p-tau)
    [1845.0, 22.4],   # Extreme low-risk (observed max amyloid + min p-tau)
]

test_labels = [
    'High-risk profile',
    'Median profile',
    'Low-risk profile',
    'Extreme AD-like',
    'Extreme low-risk',
]

test_patients = pd.DataFrame(test_array, columns=feature_cols, index=test_labels)
test_patients

,CSF Amyloid (pg/mL),CSF Phosphorylated tau (pg/mL)
High-risk profile,368.2,119.2
Median profile,604.0,63.5
Low-risk profile,1062.0,37.1
Extreme AD-like,257.0,512.0
Extreme low-risk,1845.0,22.4


In [17]:




model = joblib.load('alzheimers_progression_model.joblib')  # adjust path if needed

predictions = model.predict(test_patients)
probabilities = model.predict_proba(test_patients)[:, 1]

results = test_patients.copy()
results['predicted_progression'] = predictions
results['progression_probability'] = probabilities.round(3)

results

,CSF Amyloid (pg/mL),CSF Phosphorylated tau (pg/mL),predicted_progression,progression_probability
High-risk profile,368.2,119.2,1,0.808
Median profile,604.0,63.5,1,0.564
Low-risk profile,1062.0,37.1,0,0.175
Extreme AD-like,257.0,512.0,1,0.985
Extreme low-risk,1845.0,22.4,0,0.011
